## Import dependencies

In [1]:
from __future__ import print_function, division
from builtins import range
# Note: you may need to update your version of future
# sudo pip install -U future

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import tensorflow as tf
from sklearn.utils import shuffle

## Generate and plot the data

In [ ]:
# Generate and plot the data
N = 1000
X = np.random.random((N, 2)) * 4 - 2  # uniformly distributed between (-2, +2)
# Y = np.cos(2*X[:,0]) + np.cos(3*X[:,1]) # Makes a cosine curve
Y = X[:, 0] * X[:, 1]  # makes a saddle shape

# Plot the data
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X[:, 0], X[:, 1], Y)
plt.title("Training Data")
plt.show()

In [ ]:
# Build the neural network and train it (to fit the dataset)
D = 2 # input dimension
M = 100 # number of hidden units

# layer 1
W = np.random.randn(D, M) / np.sqrt(D)
b = np.zeros(M)

# layer 2
V = np.random.randn(M)
c = 0

# forward pass
def forward(X):
  Z = X.dot(W) + b
  Z = Z * (Z > 0) # relu

  Yhat = Z.dot(V) + c
  return Z, Yhat

# backward pass
def derivative_V(Z, Y, Yhat):
  return (Y - Yhat).dot(Z)

def derivative_c(Y, Yhat):
  return (Y - Yhat).sum()

def derivative_W(X, Z, Y, Yhat, V):
#   dZ = np.outer(Y - Yhat, V) * (Z*(1 - Z)) # this is for sigmoid
#   dZ = np.outer(Y - Yhat, V) * (1 - Z * Z) # this is for tanh
  dZ = np.outer(Y - Yhat, V) * (Z > 0) # relu
  return X.T.dot(dZ)

def derivative_b(Z, Y, Yhat, V):
  # dZ = np.outer(Y - Yhat, V) * (Z*(1 - Z)) # this is for sigmoid
  # dZ = np.outer(Y - Yhat, V) * (1 - Z * Z) # this is for tanh
  dZ = np.outer(Y - Yhat, V) * (Z > 0) # relu
  return dZ.sum(axis=0)

def update(X, Z, Y, Yhat, W, b, V, c, learning_rate=1e-4):
  dV = derivative_V(Z, Y, Yhat)
  dc = derivative_c(Y, Yhat)
  dW = derivative_W(X, Z, Y, Yhat, V)
  db = derivative_b(Z, Y, Yhat, V)

  V -= learning_rate * dV
  c -= learning_rate * dc
  W -= learning_rate * dW
  b -= learning_rate * db

  return W, b, V, c

def get_cost(Y, Yhat):
  return ((Y - Yhat)**2).mean()

# Training loop
epochs = 10000
learning_rate = 0.01
losses = []

for epoch in range(epochs):
  X, Y = shuffle(X, Y)
  Z, Yhat = forward(X)
  W, b, V, c = update(X, Z, Y, Yhat, W, b, V, c, learning_rate)
  loss = get_cost(Y, Yhat)
  losses.append(loss)

  if epoch % 1000 == 0:
    print(f'Epoch: {epoch}     Loss: {loss:.4f}')

plt.plot(losses)

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

In [ ]:
# Plot the prediction vs actual values - use subplots
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X[:,0], X[:,1], Y)
ax[0].scatter(X[:,0], X[:,1], Yhat)
ax[0].set_title('Actual vs. Predicted')
plt.grid(True)
plt.title('Training Cost')

ax[1].plot(losses)